In [ ]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

## Main Models

In [ ]:
%%capture
!pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, pipeline, logging

logging.set_verbosity_warning()

import torch
import torch.nn.functional as F

### Text Model (LLAMA 2)

In [ ]:
# %%capture
!pip install optimum>=1.12.0 accelerate optimum #transformers>=4.32.0
!pip install auto-gptq --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu118/  # Use cu117 if on CUDA 11.7

In [ ]:
%%capture
# IF YOU GET AN ERROR, RESTART RUNTIME AFTER INSTALLING RIGHT TRANSFORMERS VERSION
# To use a different branch: revision="gptq-4bit-64g-actorder_True"
llama2_model = AutoModelForCausalLM.from_pretrained("TheBloke/Llama-2-7b-Chat-GPTQ",
                                             device_map="auto",
                                             trust_remote_code=False,
                                             revision="main")

llama2_tokenizer = AutoTokenizer.from_pretrained("TheBloke/Llama-2-7b-Chat-GPTQ", use_fast=True)

llama2_tokenizer.padding_side = "left"
# llama2_tokenizer.pad_token = llama2_tokenizer.eos_token
eos_token = llama2_tokenizer(llama2_tokenizer.bos_token).input_ids[0]
llama2_tokenizer.pad_token = llama2_tokenizer.bos_token
llama2_model.config.pad_token_id = llama2_model.config.eos_token_id

In [ ]:
# prompt = "How do I greet my neighbour?"
# prompt_template=f'''[INST] <<SYS>>
# You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe.  Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature. If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share False information.
# <</SYS>>
# {prompt}[/INST]

# '''

# prompt_template2=f'''[INST]{prompt}[/INST]

# '''
# # Single inference
# input_ids = llama2_tokenizer(prompt_template2, return_tensors='pt').input_ids.cuda()
# generate_output = llama2_model.generate(inputs=input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=15, return_dict_in_generate=True, output_scores=True)
# output_ids = generate_output.sequences[0][len(input_ids[0]):]
# output_string = llama2_tokenizer.decode(generate_output.sequences[0])
# print(output_string)

# # Batch inference
# input_ids = llama2_tokenizer([prompt_template]*20, return_tensors="pt", padding=True).to(llama2_model.device)
# generate_output = llama2_model.generate(**input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
# print(llama2_tokenizer.batch_decode(generate_output.sequences, skip_special_tokens=True))

### Text Model (GPT2)

In [ ]:
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2", padding_side="left")
gpt2_model = AutoModelForCausalLM.from_pretrained("gpt2").to("cuda").eval()

gpt2_tokenizer.padding_side = "left"
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
eos_token = gpt2_tokenizer(gpt2_tokenizer.eos_token).input_ids[0]
# gpt2_tokenizer.pad_token = gpt2_tokenizer.bos_token
gpt2_model.config.pad_token_id = gpt2_model.config.eos_token_id

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# prompt = "How do I greet my neighbour?"
# prompt_template=f'''User: {prompt}
# Assistant:'''

# # Single inference
# input_ids = gpt2_tokenizer(prompt_template, return_tensors='pt').input_ids.cuda()
# generate_output = gpt2_model.generate(inputs=input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=15, return_dict_in_generate=True, output_scores=True)
# output_ids = generate_output.sequences[0][len(input_ids[0]):]
# output_string = gpt2_tokenizer.decode(generate_output.sequences[0])
# print(output_string)

# # Batch inference
# input_ids = gpt2_tokenizer([prompt_template]*20, return_tensors="pt", padding=True).to(gpt2_model.device)
# generate_output = gpt2_model.generate(**input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
# print(gpt2_tokenizer.batch_decode(generate_output.sequences, skip_special_tokens=True))

### Text Model (GPT2-Large)

In [ ]:
gpt2_large_tokenizer = AutoTokenizer.from_pretrained("gpt2-large", padding_side="left")
gpt2_large_model = AutoModelForCausalLM.from_pretrained("gpt2-large").to("cuda").eval()

gpt2_large_tokenizer.padding_side = "left"
gpt2_large_tokenizer.pad_token = gpt2_large_tokenizer.eos_token
eos_token = gpt2_large_tokenizer(gpt2_large_tokenizer.eos_token).input_ids[0]
# gpt2_large_tokenizer.pad_token = gpt2_large_tokenizer.bos_token
gpt2_large_model.config.pad_token_id = gpt2_large_model.config.eos_token_id

In [ ]:
# prompt = "How do I greet my neighbour?"
# prompt_template=f'''User: {prompt}
# Assistant:'''

# # Single inference
# input_ids = gpt2_large_tokenizer(prompt_template, return_tensors='pt').input_ids.cuda()
# generate_output = gpt2_large_model.generate(inputs=input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=15, return_dict_in_generate=True, output_scores=True)
# output_ids = generate_output.sequences[0][len(input_ids[0]):]
# output_string = gpt2_large_tokenizer.decode(generate_output.sequences[0])
# print(output_string)

# # Batch inference
# input_ids = gpt2_large_tokenizer([prompt_template]*20, return_tensors="pt", padding=True).to(gpt2_large_model.device)
# generate_output = gpt2_large_model.generate(**input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
# print(gpt2_large_tokenizer.batch_decode(generate_output.sequences, skip_special_tokens=True))

### Text Model Functions

In [ ]:
model = None
tokenizer = None
in_batch_size = None
prefix = None
remove_prefix = None
vocab_size = None
prompt_template = None

def set_model_params(model_name):
    global model, tokenizer, in_batch_size, prefix, remove_prefix, vocab_size, prompt_template

    if model_name == "gpt2":
        model = gpt2_model
        tokenizer = gpt2_tokenizer
        in_batch_size = 200#400
        prefix = " "
        remove_prefix = 0
        vocab_size = 50257
        prompt_template = lambda x: f"User: {x}\nAssistant:"

    elif model_name == "llama2":
        model = llama2_model
        tokenizer = llama2_tokenizer
        in_batch_size = 200
        prefix = "\n\n"
        remove_prefix = 4
        vocab_size = 32000
        prompt_template = lambda x: f"[INST]{x}[/INST]\n\n"


In [ ]:
def fluent_model_generate(seed=None, num_generate=1, batch_size=100, max_new_tokens=20):
    if seed is None:
        prefix_length = 2
    else:
        if seed[-1] == " ":
            seed = seed[:-1]
        if seed[-1] not in [".", "!", "?"]:
            seed += "."
        prefix_length = len(tokenizer(seed).input_ids)

    all_sequences = []
    for i in range(0, num_generate, batch_size):
        batch_s = min(batch_size, num_generate-i)
        if seed is None:
            prompt_batch = [tokenizer.decode(random.randint(0,vocab_size))+". " for _ in range(batch_s)]
        else:
            prompt_batch = [seed]*batch_s

        input_ids = tokenizer(prompt_batch, return_tensors="pt", padding=True).to(model.device)
        generate_output = model.generate(**input_ids, temperature=1, do_sample=True, top_p=0.95, top_k=100, max_new_tokens=max_new_tokens, return_dict_in_generate=True, output_scores=True, pad_token_id=eos_token)
        sequences = [seq[prefix_length:] for seq in generate_output.sequences]
        all_sequences += sequences

    random_tokens = lambda : tokenizer.decode([random.randint(0, vocab_size) for _ in range(random.randint(1, max_new_tokens))])
    # random_tokens = lambda : "None"

    decoded_sequences = tokenizer.batch_decode(all_sequences, skip_special_tokens=True)
    decoded_sequences = [random_tokens() if len(seq) == 0 else seq for seq in decoded_sequences]
    decoded_sequences = [seq[1:] if seq[0] in [" ", "\xa0", "\n"] else seq for seq in decoded_sequences]
    decoded_sequences = [seq[1:] if seq[0] in [" ", "\xa0", "\n"] else seq for seq in decoded_sequences]
    # decoded_sequences = [seq.replace("\xa0", "") for seq in decoded_sequences]

    return decoded_sequences

In [ ]:
# Find probability of any sequence given context
def prob_of_sequence(context, test_seq):
    # prompt_template = lambda x: f"User: {x}\nAssistant:"

    test_string = " " + test_seq
    test_tokens = tokenizer(test_string, return_tensors='pt').input_ids[0]
    # test_tokens = test_tokens[4:] # Remove '<s> \n\n'

    current_toks = []
    seq_prob = 1
    for t_tok in test_tokens:
        current_context = context + tokenizer.decode(current_toks)
        new_input_ids = tokenizer(current_context, return_tensors='pt').input_ids.cuda()
        new_generate_output = model.generate(inputs=new_input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=300, max_new_tokens=1, return_dict_in_generate=True, output_scores=True, pad_token_id=eos_token,)
        new_generate_output_scores = F.softmax(new_generate_output.scores[0][0], dim=0)
        t_score = new_generate_output_scores[t_tok]

        # Just for test, distributions
#         distribution = []
#         nonzero_indices = torch.nonzero(new_generate_output_scores)
#         nonzero_values = new_generate_output_scores[nonzero_indices[:, 0]]
#         sorted_tensor = torch.sort(nonzero_values, descending=True)
#         sorted_indices = nonzero_indices[sorted_tensor.indices]
#         for i in range(len(sorted_indices)):
#             distribution.append([tokenizer.decode(sorted_indices[i][0]), int(sorted_indices[i][0]), round(float(sorted_tensor.values[i]),4)])
#         print(tokenizer.decode(t_tok), "\t-", distribution)

        if t_score == 0:
            # print("\nProbability of sequence: 0 %")
            return 0

        seq_prob *= float(t_score)
        current_toks.append(t_tok)

    # print("Probability of sequence:", seq_prob * 100, "%")
    return seq_prob

### Embedding Model

In [ ]:
%%capture
!pip install sentencepiece accelerate

In [ ]:
class BottleneckT5Autoencoder:
    def __init__(self, model_path: str, device='cpu'):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)#, model_max_length=512)
        self.model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True).to(self.device)
        self.model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
autoencoder = BottleneckT5Autoencoder(model_path='thesephist/contra-bottleneck-t5-small-wikipedia', device=device)
embedding_size = 512

In [ ]:
t5_dummy_tokens = autoencoder.tokenizer(".", return_tensors='pt').to(device)['input_ids']

def t5_encode(text, t5_decoder_inputs=t5_dummy_tokens):
    with torch.no_grad():
        inputs = autoencoder.tokenizer(text, return_tensors='pt').to(device)#, padding=True, truncation=True).to(device)
        outputs = autoencoder.model(**inputs, decoder_input_ids=t5_decoder_inputs, encode_only=True)
        # torch.cuda.empty_cache()
        del inputs # cleaning
        return outputs

def t5_encode_list(texts):
    embeddings = []
    for text in texts:
        emb = t5_encode(text)#[0]
        embeddings.append(emb)
    return torch.stack(embeddings, dim=0)

t5_dummy_embed = t5_encode(".")

def t5_decode(embeddings, t5_batch_size, max_out_length=15, t5_dummy_tokens=t5_dummy_tokens, t5_dummy_embed=t5_dummy_embed, sample=False):
    with torch.no_grad():
        decoded_output = []
        for batch in range(0,len(embeddings),t5_batch_size):
            # if t5_batch_size != 1:
                # print(batch+t5_batch_size, ",", end="")
            current_embeddings = embeddings[batch:batch+t5_batch_size]
            autoencoder.model.perturb_vector = current_embeddings - t5_dummy_embed.unsqueeze(0)
            input_ids = t5_dummy_tokens.repeat(current_embeddings.shape[0], 1)  # Repeat the dummy text for each latent
            if not sample:
                outputs = autoencoder.model.generate(
                    input_ids=input_ids,
                    max_length=max_out_length,
                    do_sample=True,
                    temperature=1.0,
                    top_p=0.9,
                    num_return_sequences=1
                )
            else:
                outputs = autoencoder.model.generate(
                    input_ids=input_ids,
                    max_length=max_out_length,
                    do_sample=True,
                    temperature=1.0,
                    num_return_sequences=1
                )
            decoded_output += [autoencoder.tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
            del current_embeddings # cleaning
            del input_ids # cleaning
            del outputs # cleaning
        # torch.cuda.empty_cache()
        return decoded_output

In [ ]:
# texts = [
#     'The quick brown fox jumps over the lazy dog',
#     'The universe isdfr no obligation to make sense to you.',
#     'whats going on you crazy bastard?',
#     'Notion is a single space where you can think, write, and plan. Capture thoughts, manage projects, or even run an entire company — and do it exactly the way you want.',
#     ]

# latents = t5_encode_list(texts)
# t5_decode(latents, max_out_length=999, sample=False, t5_batch_size=5)

In [ ]:
# latents = torch.rand((4, 1, 512)).to(device)
# latents = (latents *2) -1
# t5_decode(latents, max_out_length=999, sample=True)

In [ ]:
# t5_tokenizer = AutoTokenizer.from_pretrained("thesephist/contra-bottleneck-t5-small-wikipedia", model_max_length=embedding_size, padding_side='left')
# t5_model = AutoModelForCausalLM.from_pretrained("thesephist/contra-bottleneck-t5-small-wikipedia", trust_remote_code=True).to("cuda").eval()

# def t5_encode(prompt):
#     with torch.no_grad():
#         inputs = t5_tokenizer(prompt, return_tensors="pt").to("cuda")
#         # Need to use dummy input for encoding
#         dummy_inputs = t5_tokenizer("", return_tensors="pt").to("cuda")['input_ids']
#         encoding = t5_model(**inputs, decoder_input_ids=dummy_inputs, encode_only=True)[0]
#         return encoding

# # Need to use another dummy input for decoding
# dummy_inputs2 = t5_tokenizer(".", return_tensors="pt").to("cuda")['input_ids']
# dummy_encoding = t5_encode('.')
# def t5_decode(embedding, max_out_length=15):
#     with torch.no_grad():
#         t5_model.perturb_vector = embedding - dummy_encoding
#         output = t5_model.generate(input_ids=dummy_inputs2, max_length=max_out_length, do_sample=True, temperature=1, top_p=0.9, num_return_sequences=1)
#         return t5_tokenizer.decode(output[0], skip_special_tokens=True)

## Initialisation

### Inversion Model

In [ ]:
inv_model = AutoModelForSeq2SeqLM.from_pretrained("wentingzhao/inversion_seq2seq_baseline").eval().to("cuda")
inv_tokenizer = AutoTokenizer.from_pretrained("wentingzhao/inversion_seq2seq_baseline")

inv_tokenizer.padding_side = "left"
inv_tokenizer.pad_token = inv_tokenizer.eos_token
inv_model.config.pad_token_id = inv_model.config.eos_token_id
inv_eos_token = inv_tokenizer(inv_tokenizer.eos_token).input_ids[0]

inv_model = None
torch.cuda.empty_cache()
del inv_model
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [ ]:
def inversion_model_generate(target_output, num_inversions=1, batch_size=100, max_new_tokens=20, sample=False):
    # Load model
    inv_model = AutoModelForSeq2SeqLM.from_pretrained("wentingzhao/inversion_seq2seq_baseline").eval().to("cuda")

    outputs = []
    for i in range(0, num_inversions, batch_size):
        batch_s = min(batch_size, num_inversions-i)
        input_ids = inv_tokenizer([target_output]*batch_s, return_tensors="pt", padding=True).input_ids.to("cuda")

        if sample:
            output = inv_model.generate(
                inputs=input_ids,
                temperature=1,
                do_sample=True,
                top_p=0.99,
                top_k=500,
                max_new_tokens=max_new_tokens,
                pad_token_id=inv_eos_token,
            )
        else:
            output = inv_model.generate(
                inputs=input_ids,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=inv_eos_token,
            )
        outputs += list(output)

    # Delete model
    inv_model = None
    torch.cuda.empty_cache()
    del inv_model
    torch.cuda.empty_cache()

    return [inv_tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [ ]:
# inversion_model_generate("Blake", num_inversions=10, max_new_tokens=20, sample=True)

### Paraphrase Model

In [ ]:
paraph_model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws").eval().to('cuda')
paraph_tokenizer = AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws")

paraph_model = None
torch.cuda.empty_cache()
del paraph_model
torch.cuda.empty_cache()

In [ ]:
def paraphrase_model_generate(target_output, num_paraphrase=1, batch_size=100, max_new_tokens=20):
    # Load model
    paraph_model = AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws").eval().to('cuda')

    text =  "paraphrase: " + target_output + " </s>"
    encoding = paraph_tokenizer.encode_plus(text, padding='longest', return_tensors="pt")
    input_ids, attention_masks = encoding["input_ids"].to("cuda"), encoding["attention_mask"].to("cuda")

    outputs = []
    for i in range(0, num_paraphrase, batch_size):
        batch_s = min(batch_size, num_paraphrase-i)

        output = paraph_model.generate(
            input_ids=input_ids,
            attention_mask=attention_masks,
            do_sample=True,
            temperature=1.5,
            top_k=500,
            top_p=0.99,
            # early_stopping=True,
            max_length=max_new_tokens,
            num_return_sequences=batch_s
        )
        outputs += list(output)

    # Delete model
    paraph_model = None
    torch.cuda.empty_cache()
    del paraph_model
    torch.cuda.empty_cache()

    return [paraph_tokenizer.decode(outputs[i], skip_special_tokens=True, clean_up_tokenization_spaces=True) for i in range(len(outputs))]

In [ ]:
# paraphrase_model_generate("This is something which i cannot understand at all", num_paraphrase=1000, batch_size=20, max_new_tokens=20)

### Synonym Function

In [ ]:
import nltk
from nltk.corpus import wordnet
import random

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name()
            synonyms.add(synonym)
    return list(synonyms)

def synonyms_model_generate(target_output, num_synonyms=1, max_new_tokens=20):
    words = target_output.split()
    variations = []
    for _ in range(num_synonyms):
        new_sentence = []
        for word in words:
            synonyms = get_synonyms(word)
            if synonyms:
                new_sentence.append(random.choice(synonyms).replace("_", " "))
            else:
                new_sentence.append(word)
        variations.append(' '.join(new_sentence[:max_new_tokens]))
    return variations

In [ ]:
# synonyms_model_generate("The quick brown fox jumped over the lazy dog", num_synonyms=10, max_new_tokens=5)

### Random Data Function

In [ ]:
!cp "/content/drive/MyDrive/University/Year 4/LLM/Saves/dataset_tweet-scraper_2024-03-04_15-49-10-416.json" "/content/"

In [ ]:
import json
import random

with open("dataset_tweet-scraper_2024-03-04_15-49-10-416.json", encoding="utf-8") as f:
    ood_data = json.load(f)

In [ ]:
def rand_data_model_generate(num_data=1, max_new_tokens=20):
    ood_sample = random.sample(ood_data, num_data)
    ood_sample = [tokenizer.decode(tokenizer(sample["text"]).input_ids[:max_new_tokens]) for sample in ood_sample]
    return ood_sample

In [ ]:
# rand_data_model_generate(num_data=30, max_new_tokens=20)

## Datasets

### Chatbot arena

In [ ]:
# !pip install huggingface_hub
# from huggingface_hub import notebook_login
# notebook_login()
# # <your Hugging Face token>

In [ ]:
%%capture
!pip install datasets
from datasets import load_dataset, load_from_disk

# dataset = load_dataset("lmsys/chatbot_arena_conversations", split="train")

In [ ]:
# print("Length of full dataset:", len(dataset))

# num_data = 100
# def conditions(x):
#     # winner_vicuna = (x["winner"] == "model_a" and x["model_a"] == "vicuna-13b")
#     # winner_vicuna = winner_vicuna or (x["winner"] == "model_b" and x["model_b"] == "vicuna-13b")
#     # if not winner_vicuna:
#     #     return False

#     # if not x["winner"] != "tie (bothbad)":
#     #     return False

#     if not  x["language"] == "English":
#         return False

#     if not x["turn"] == 1:
#         return False

#     # no_harm1 = not x["toxic_chat_tag"]["roberta-large"]["flagged"]
#     # no_harm1 = no_harm1 and not x["toxic_chat_tag"]["t5-large"]["flagged"]
#     # if not no_harm1:
#     #     return False

#     # # no_harm2
#     # for is_harmful in x["openai_moderation"]["categories"].values():
#     #     if is_harmful:
#     #         return False

#     convo = "conversation_a" if x["winner"] == "model_a" else "conversation_b"
#     max_len = len(tokenizer(x[convo][0]['content']).input_ids) <= 64#15
#     max_len = max_len and len(tokenizer(x[convo][1]['content']).input_ids) <= 100
#     if not max_len:
#         return False

#     inputt = prompt_template(x[convo][0]['content'])
#     outputt = x[convo][1]['content']
#     if prob_of_sequence(inputt, outputt) == 0:
#         return False

#     return True

# # small_dataset = dataset.select([0, 10, 20, 30, 40, 50])
# # even_dataset = dataset.filter(lambda example, idx: idx % 2 == 0, with_indices=True)

# processed_data = dataset.filter(lambda x: conditions(x))
# processed_data = processed_data.select_columns(["question_id", "winner", "conversation_a", "conversation_b"])
# processed_data = processed_data.shuffle(seed=42)
# print("Length after processing:", len(processed_data))

In [ ]:
# reloaded_data = load_from_disk("processed_dataset")
# processed_data = reloaded_data[:30]

reloaded_data = load_from_disk("processed_dataset2")
processed_data = reloaded_data[:50]#[:226]

print("Length of chosen subset:", len(processed_data["question_id"]))
user_text = lambda x: processed_data["conversation_a" if processed_data["winner"][x] == "model_a" else "conversation_b"][x][0]['content']
assistant_text = lambda x: processed_data["conversation_a" if processed_data["winner"][x] == "model_a" else "conversation_b"][x][1]['content']
# print("\nExample:")
# print([user_text(0)])
# print([assistant_text(0)])

Length of chosen subset: 50


## Algorithms

### Text-based Optimisation [Genetic (using DEAP)]

In [ ]:
%%capture
!pip install deap

In [ ]:
import random
import numpy
import pickle

from deap import base, creator, tools #, algorithms

import math
import torch.nn.functional as F
import copy
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
import time


#### Set up

##### Simple algo definitions

In [ ]:
def varAnd(population, toolbox, cxpb, mutpb):
    offspring = [toolbox.clone(ind) for ind in population]

    # Apply crossover and mutation on the offspring
    for i in range(1, len(offspring), 2):
        if random.random() < cxpb:
            offspring[i - 1], offspring[i] = toolbox.mate(offspring[i - 1],
                                                          offspring[i])
            del offspring[i - 1].fitness.values, offspring[i].fitness.values

    for i in range(len(offspring)):
        if random.random() < mutpb:
            offspring[i], = toolbox.mutate(offspring[i])
            del offspring[i].fitness.values

    return offspring



def eaSimple(population, toolbox, cxpb, mutpb, ngen, stats=None, halloffame=None, verbose=__debug__,
             gens_so_far=0, experiment_start_time=None, max_minutes=10, stop_early=False):
    global using_main_model

    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals', 'time'] + (stats.fields if stats else [])

    # # Evaluate the individuals with an invalid fitness
    # invalid_ind = [ind for ind in population if not ind.fitness.valid]
    # # fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
    # fitnesses = toolbox.evaluate(ind_list=invalid_ind)
    # for ind, fit in zip(invalid_ind, fitnesses):
    #     ind.fitness.values = fit

    if halloffame is not None:
        halloffame.update(population)

    # record = stats.compile(population) if stats else {}
    # logbook.record(gen=0, nevals=len(invalid_ind), **record)
    # if verbose:
    #     print(logbook.stream)

    # Begin the generational process
    for gen in range(1, ngen + 1):
        # Select the next generation individuals
        offspring = toolbox.select(population, len(population))

        # Vary the pool of individuals
        offspring = varAnd(offspring, toolbox, cxpb, mutpb)

        # Evaluate the individuals with an invalid fitness
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        # fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        fitnesses = toolbox.evaluate(ind_list=invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        # Update the hall of fame with the generated individuals
        if halloffame is not None:
            halloffame.update(offspring)

        # Replace the current population by the offspring
        population[:] = offspring

        # Append the current generation statistics to the logbook
        record = stats.compile(population) if stats else {}
        # logbook.record(gen=gen, nevals=len(invalid_ind), **record)
        logbook.record(gen=(gen+gens_so_far), nevals=len(invalid_ind), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **record)
        # if verbose:
            # print(logbook.stream)
            # if gen % 10 == 0:
            # if max([ind.fitness.values[0] for ind in population]) > 0:
            # print("Best so far:", [tokenizer.decode(tools.selBest(population, k=1)[0])])

        # Print best ones
        print(logbook.__str__().split("\n")[-1])
        best = tools.selBest(population, k=1)#000)
        best_fitnesses = [ind.fitness.values[0] for ind in best]
        print([[best_fitnesses[i], [tokenizer.decode(best[i]).replace("'", "\\'").replace('"', '\\"')]] for i in range(len(best))])

        # Check if should be stopping
        if stop_early and best_fitnesses[0] == 1:
            return population, logbook, True
        # if timedelta(seconds=(time.process_time()-experiment_start_time)) >= timedelta(minutes=max_minutes):
        #     return population, logbook, True

        # Check if should be switching from intial model to main model, if havent already
        if not using_main_model:
            if initial_model_based_on == "gens":
                if (gen+gens_so_far) >= initial_model_param:
                    set_model_params(main_model_name)
                    using_main_model = True
            elif initial_model_based_on == "max_score":
                if best_fitnesses[0] >= initial_model_param:
                    set_model_params(main_model_name)
                    using_main_model = True

    return population, logbook, False


##### ~Complex algo definitions~

In [ ]:
# def varOr(population, toolbox, lambda_, cxpb, mutpb):
#     r"""Part of an evolutionary algorithm applying only the variation part
#     (crossover, mutation **or** reproduction). The modified individuals have
#     their fitness invalidated. The individuals are cloned so returned
#     population is independent of the input population.

#     :param population: A list of individuals to vary.
#     :param toolbox: A :class:`~deap.base.Toolbox` that contains the evolution
#                     operators.
#     :param lambda\_: The number of children to produce
#     :param cxpb: The probability of mating two individuals.
#     :param mutpb: The probability of mutating an individual.
#     :returns: The final population.

#     The variation goes as follow. On each of the *lambda_* iteration, it
#     selects one of the three operations; crossover, mutation or reproduction.
#     In the case of a crossover, two individuals are selected at random from
#     the parental population :math:`P_\mathrm{p}`, those individuals are cloned
#     using the :meth:`toolbox.clone` method and then mated using the
#     :meth:`toolbox.mate` method. Only the first child is appended to the
#     offspring population :math:`P_\mathrm{o}`, the second child is discarded.
#     In the case of a mutation, one individual is selected at random from
#     :math:`P_\mathrm{p}`, it is cloned and then mutated using using the
#     :meth:`toolbox.mutate` method. The resulting mutant is appended to
#     :math:`P_\mathrm{o}`. In the case of a reproduction, one individual is
#     selected at random from :math:`P_\mathrm{p}`, cloned and appended to
#     :math:`P_\mathrm{o}`.

#     This variation is named *Or* because an offspring will never result from
#     both operations crossover and mutation. The sum of both probabilities
#     shall be in :math:`[0, 1]`, the reproduction probability is
#     1 - *cxpb* - *mutpb*.
#     """
#     assert (cxpb + mutpb) <= 1.0, (
#         "The sum of the crossover and mutation probabilities must be smaller "
#         "or equal to 1.0.")

#     offspring = []
#     for _ in range(lambda_):
#         op_choice = random.random()
#         if op_choice < cxpb:            # Apply crossover
#             ind1, ind2 = [toolbox.clone(i) for i in random.sample(population, 2)]
#             ind1, ind2 = toolbox.mate(ind1, ind2)
#             del ind1.fitness.values
#             offspring.append(ind1)
#         elif op_choice < cxpb + mutpb:  # Apply mutation
#             ind = toolbox.clone(random.choice(population))
#             ind, = toolbox.mutate(ind)
#             del ind.fitness.values
#             offspring.append(ind)
#         else:                           # Apply reproduction
#             offspring.append(random.choice(population))

#     return offspring



# def eaMuPlusLambda(population, toolbox, mu, lambda_, cxpb, mutpb, ngen,
#                    stats=None, halloffame=None, verbose=__debug__):
#     r"""This is the :math:`(\mu + \lambda)` evolutionary algorithm.

#     :param population: A list of individuals.
#     :param toolbox: A :class:`~deap.base.Toolbox` that contains the evolution
#                     operators.
#     :param mu: The number of individuals to select for the next generation.
#     :param lambda\_: The number of children to produce at each generation.
#     :param cxpb: The probability that an offspring is produced by crossover.
#     :param mutpb: The probability that an offspring is produced by mutation.
#     :param ngen: The number of generation.
#     :param stats: A :class:`~deap.tools.Statistics` object that is updated
#                   inplace, optional.
#     :param halloffame: A :class:`~deap.tools.HallOfFame` object that will
#                        contain the best individuals, optional.
#     :param verbose: Whether or not to log the statistics.
#     :returns: The final population
#     :returns: A class:`~deap.tools.Logbook` with the statistics of the
#               evolution.

#     The algorithm takes in a population and evolves it in place using the
#     :func:`varOr` function. It returns the optimized population and a
#     :class:`~deap.tools.Logbook` with the statistics of the evolution. The
#     logbook will contain the generation number, the number of evaluations for
#     each generation and the statistics if a :class:`~deap.tools.Statistics` is
#     given as argument. The *cxpb* and *mutpb* arguments are passed to the
#     :func:`varOr` function. The pseudocode goes as follow ::

#         evaluate(population)
#         for g in range(ngen):
#             offspring = varOr(population, toolbox, lambda_, cxpb, mutpb)
#             evaluate(offspring)
#             population = select(population + offspring, mu)

#     First, the individuals having an invalid fitness are evaluated. Second,
#     the evolutionary loop begins by producing *lambda_* offspring from the
#     population, the offspring are generated by the :func:`varOr` function. The
#     offspring are then evaluated and the next generation population is
#     selected from both the offspring **and** the population. Finally, when
#     *ngen* generations are done, the algorithm returns a tuple with the final
#     population and a :class:`~deap.tools.Logbook` of the evolution.

#     This function expects :meth:`toolbox.mate`, :meth:`toolbox.mutate`,
#     :meth:`toolbox.select` and :meth:`toolbox.evaluate` aliases to be
#     registered in the toolbox. This algorithm uses the :func:`varOr`
#     variation.
#     """
#     logbook = tools.Logbook()
#     logbook.header = ['gen', 'nevals'] + (stats.fields if stats else [])

#     # Evaluate the individuals with an invalid fitness
#     invalid_ind = [ind for ind in population if not ind.fitness.valid]
#     fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
#     for ind, fit in zip(invalid_ind, fitnesses):
#         ind.fitness.values = fit

#     if halloffame is not None:
#         halloffame.update(population)

#     record = stats.compile(population) if stats is not None else {}
#     logbook.record(gen=0, nevals=len(invalid_ind), **record)
#     if verbose:
#         print(logbook.stream)

#     # Begin the generational process
#     for gen in range(1, ngen + 1):
#         # Vary the population
#         offspring = varOr(population, toolbox, lambda_, cxpb, mutpb)

#         # Evaluate the individuals with an invalid fitness
#         invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
#         fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
#         for ind, fit in zip(invalid_ind, fitnesses):
#             ind.fitness.values = fit

#         # Update the hall of fame with the generated individuals
#         if halloffame is not None:
#             halloffame.update(offspring)

#         # Select the next generation population
#         population[:] = toolbox.select(population + offspring, mu)

#         # Update the statistics with the new population
#         record = stats.compile(population) if stats is not None else {}
#         logbook.record(gen=gen, nevals=len(invalid_ind), **record)
#         if verbose:
#             print(logbook.stream)

#     return population, logbook



# def eaMuCommaLambda(population, toolbox, mu, lambda_, cxpb, mutpb, ngen,
#                     stats=None, halloffame=None, verbose=__debug__):
#     r"""This is the :math:`(\mu~,~\lambda)` evolutionary algorithm.

#     :param population: A list of individuals.
#     :param toolbox: A :class:`~deap.base.Toolbox` that contains the evolution
#                     operators.
#     :param mu: The number of individuals to select for the next generation.
#     :param lambda\_: The number of children to produce at each generation.
#     :param cxpb: The probability that an offspring is produced by crossover.
#     :param mutpb: The probability that an offspring is produced by mutation.
#     :param ngen: The number of generation.
#     :param stats: A :class:`~deap.tools.Statistics` object that is updated
#                   inplace, optional.
#     :param halloffame: A :class:`~deap.tools.HallOfFame` object that will
#                        contain the best individuals, optional.
#     :param verbose: Whether or not to log the statistics.
#     :returns: The final population
#     :returns: A class:`~deap.tools.Logbook` with the statistics of the
#               evolution

#     The algorithm takes in a population and evolves it in place using the
#     :func:`varOr` function. It returns the optimized population and a
#     :class:`~deap.tools.Logbook` with the statistics of the evolution. The
#     logbook will contain the generation number, the number of evaluations for
#     each generation and the statistics if a :class:`~deap.tools.Statistics` is
#     given as argument. The *cxpb* and *mutpb* arguments are passed to the
#     :func:`varOr` function. The pseudocode goes as follow ::

#         evaluate(population)
#         for g in range(ngen):
#             offspring = varOr(population, toolbox, lambda_, cxpb, mutpb)
#             evaluate(offspring)
#             population = select(offspring, mu)

#     First, the individuals having an invalid fitness are evaluated. Second,
#     the evolutionary loop begins by producing *lambda_* offspring from the
#     population, the offspring are generated by the :func:`varOr` function. The
#     offspring are then evaluated and the next generation population is
#     selected from **only** the offspring. Finally, when
#     *ngen* generations are done, the algorithm returns a tuple with the final
#     population and a :class:`~deap.tools.Logbook` of the evolution.

#     .. note::

#         Care must be taken when the lambda:mu ratio is 1 to 1 as a
#         non-stochastic selection will result in no selection at all as the
#         operator selects *lambda* individuals from a pool of *mu*.


#     This function expects :meth:`toolbox.mate`, :meth:`toolbox.mutate`,
#     :meth:`toolbox.select` and :meth:`toolbox.evaluate` aliases to be
#     registered in the toolbox. This algorithm uses the :func:`varOr`
#     variation.
#     """
#     assert lambda_ >= mu, "lambda must be greater or equal to mu."

#     # Evaluate the individuals with an invalid fitness
#     invalid_ind = [ind for ind in population if not ind.fitness.valid]
#     fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
#     for ind, fit in zip(invalid_ind, fitnesses):
#         ind.fitness.values = fit

#     if halloffame is not None:
#         halloffame.update(population)

#     logbook = tools.Logbook()
#     logbook.header = ['gen', 'nevals'] + (stats.fields if stats else [])

#     record = stats.compile(population) if stats is not None else {}
#     logbook.record(gen=0, nevals=len(invalid_ind), **record)
#     if verbose:
#         print(logbook.stream)

#     # Begin the generational process
#     for gen in range(1, ngen + 1):
#         # Vary the population
#         offspring = varOr(population, toolbox, lambda_, cxpb, mutpb)

#         # Evaluate the individuals with an invalid fitness
#         invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
#         fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
#         for ind, fit in zip(invalid_ind, fitnesses):
#             ind.fitness.values = fit

#         # Update the hall of fame with the generated individuals
#         if halloffame is not None:
#             halloffame.update(offspring)

#         # Select the next generation population
#         population[:] = toolbox.select(offspring, mu)

#         # Update the statistics with the new population
#         record = stats.compile(population) if stats is not None else {}
#         logbook.record(gen=gen, nevals=len(invalid_ind), **record)
#         if verbose:
#             print(logbook.stream)
#     return population, logbook



# def eaGenerateUpdate(toolbox, ngen, halloffame=None, stats=None,
#                      verbose=__debug__):
#     """This is algorithm implements the ask-tell model proposed in
#     [Colette2010]_, where ask is called `generate` and tell is called `update`.

#     :param toolbox: A :class:`~deap.base.Toolbox` that contains the evolution
#                     operators.
#     :param ngen: The number of generation.
#     :param stats: A :class:`~deap.tools.Statistics` object that is updated
#                   inplace, optional.
#     :param halloffame: A :class:`~deap.tools.HallOfFame` object that will
#                        contain the best individuals, optional.
#     :param verbose: Whether or not to log the statistics.
#     :returns: The final population
#     :returns: A class:`~deap.tools.Logbook` with the statistics of the
#               evolution

#     The algorithm generates the individuals using the :func:`toolbox.generate`
#     function and updates the generation method with the :func:`toolbox.update`
#     function. It returns the optimized population and a
#     :class:`~deap.tools.Logbook` with the statistics of the evolution. The
#     logbook will contain the generation number, the number of evaluations for
#     each generation and the statistics if a :class:`~deap.tools.Statistics` is
#     given as argument. The pseudocode goes as follow ::

#         for g in range(ngen):
#             population = toolbox.generate()
#             evaluate(population)
#             toolbox.update(population)


#     This function expects :meth:`toolbox.generate` and :meth:`toolbox.evaluate` aliases to be
#     registered in the toolbox.

#     .. [Colette2010] Collette, Y., N. Hansen, G. Pujol, D. Salazar Aponte and
#        R. Le Riche (2010). On Object-Oriented Programming of Optimizers -
#        Examples in Scilab. In P. Breitkopf and R. F. Coelho, eds.:
#        Multidisciplinary Design Optimization in Computational Mechanics,
#        Wiley, pp. 527-565;

#     """
#     logbook = tools.Logbook()
#     logbook.header = ['gen', 'nevals'] + (stats.fields if stats else [])

#     for gen in range(ngen):
#         # Generate a new population
#         population = toolbox.generate()
#         # Evaluate the individuals
#         fitnesses = toolbox.map(toolbox.evaluate, population)
#         for ind, fit in zip(population, fitnesses):
#             ind.fitness.values = fit

#         if halloffame is not None:
#             halloffame.update(population)

#         # Update the strategy with the evaluated individuals
#         toolbox.update(population)

#         record = stats.compile(population) if stats is not None else {}
#         logbook.record(gen=gen, nevals=len(population), **record)
#         if verbose:
#             print(logbook.stream)

#     return population, logbook

##### Evaluation function

In [ ]:
def eval_inds(ind_list, in_batch_size, target_tokens, prompt_template=None, target_tok_id=0):
    # Have to convert to list of strings to stay same with previous code
    string_list = tokenizer.batch_decode(ind_list) #, skip_special_tokens=True)
    if prompt_template is not None:
        string_list = [prompt_template(s) for s in string_list]

    num_strings = len(string_list)
    batch_size = min(in_batch_size, num_strings)
    # Initialise batch
    batch_string_ids = list(range(0,batch_size))
    next_string_to_batch = batch_size

    # Initialise string trackers
    if target_tok_id == 0:
        string_probabilities = [1]*num_strings
        string_target_tok_id = [0]*num_strings
    else:
        # Individuals have already gotten probabilities for some tokens
        string_probabilities = [ind.fitness.values[0] if ind.fitness.valid else 1 for ind in ind_list]
        string_target_tok_id = [target_tok_id if ind.fitness.valid else 0 for ind in ind_list]

    successful_string_ids = []

    # Go until there are no more strings being checked nor yet to be checked
    empty_batch_slots = 0
    counter = 0
    while empty_batch_slots != batch_size:
        # Get batch ready for inference
        batch_strings = []
        for b_id in range(batch_size):
            if b_id >= num_strings:
                break
            s_id = batch_string_ids[b_id]
            if s_id == -1:
                continue
            # Need to add target tokens checked so far to end of string
            b_string = string_list[batch_string_ids[b_id]]
            b_string += tokenizer.decode(target_tokens[:string_target_tok_id[s_id]])
            batch_strings.append(b_string)

        # Do the inference
        input_ids = tokenizer(batch_strings, return_tensors="pt", padding=True).to("cuda")
        generate_output = model.generate(**input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=300, max_new_tokens=1, return_dict_in_generate=True, output_scores=True, pad_token_id=eos_token,)
        # generate_output = model.generate(**input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)

        # Update validity of strings based on inference
        empty = 0
        for b_id in range(batch_size):
            s_id = batch_string_ids[b_id]
            if s_id == -1:
                empty += 1
                continue
            curr_target_tok = target_tokens[string_target_tok_id[s_id]]
            prob = generate_output.scores[0][b_id-empty][curr_target_tok]

            if not (math.isinf(prob) and prob < 0):
                # String still viable candidate
                # Update probability
                True_distribution = F.softmax(generate_output.scores[0][b_id-empty], dim=0)
                string_probabilities[s_id] *= float(True_distribution[curr_target_tok])

                # Prepare to check for next target token
                string_target_tok_id[s_id] += 1
                if string_target_tok_id[s_id] == len(target_tokens):
                    # Have already checked all the target tokens
                    successful_string_ids.append(s_id)
                    # Finished so pretend string is not viable so it gets replaced
                    prob = -math.inf
            else:
                # String not a viable candidate
                # Update probability
                string_probabilities[s_id] = 0

            if math.isinf(prob) and prob < 0:
                # Replace non-viable and finished strings in batch
                if next_string_to_batch >= num_strings:
                    # No new strings to start checking
                    batch_string_ids[b_id] = -1
                    empty_batch_slots += 1
                else:
                    batch_string_ids[b_id] = next_string_to_batch
                    next_string_to_batch += 1
        counter += 1

    ind_probabilities = []
    for prob in string_probabilities:
        ind_probabilities.append((prob,))
    return ind_probabilities

##### Mutate function

In [ ]:
def custom_mutate(individual, indpb, bounded):
    # Keep track of elements index offset since inserts and deletes move it around
    offset = 0
    for i in range(len(individual)):
        # Each element has independent probability of recieving mutation
        if random.random() >= indpb:
            continue

        # Which mutations can be performed depends on the length of the individual
        types = [1,2,3,4]
        if len(individual) == 0:
            types.remove(1) # modify
            types.remove(3) # delete
            types.remove(4) # swap
        elif len(individual) == 1:
            types.remove(3) # delete
            types.remove(4) # swap
        elif bounded and len(individual) == max_input_len:
            types.remove(2) # insert

        # Apply just one of the available mutations to this element
        choice = random.choice(types)
        ind = i + offset
        if choice == 1: # modify
            individual[ind] = random.randint(0, vocab_size-1)
        elif choice == 2: # insert
            individual.insert(ind, random.randint(0, vocab_size-1))
            offset += 1
        elif choice == 3: # delete
            del individual[ind]
            offset -= 1
        elif choice == 4: # swap
            ind2 = (ind + random.randint(1,len(individual)-1)) % len(individual)
            temp = individual[ind]
            individual[ind] = individual[ind2]
            individual[ind2] = temp

    return individual,

##### Initialisation function

In [ ]:
def initialise_text_pop(start_population_as, sample_x, max_input_len, num_inputs):
    if start_population_as == "random":
        pop = [toolbox.individual(n=random.randint(1,max_input_len)) for _ in range(num_inputs)]
    elif start_population_as == "random_dataset":
        random_dataset = rand_data_model_generate(num_data=num_inputs, max_new_tokens=max_input_len)
        pop = [creator.Individual(tokenizer(random_dataset[i]).input_ids) for i in range(num_inputs)]
    elif start_population_as == "random_fluent":
        random_fluent = fluent_model_generate(num_generate=num_inputs, max_new_tokens=max_input_len)
        pop = [creator.Individual(tokenizer(random_fluent[i]).input_ids) for i in range(num_inputs)]
    elif start_population_as == "random_fluent_output":
        random_fluent_output = fluent_model_generate(seed=assistant_text(sample_x), num_generate=num_inputs, max_new_tokens=max_input_len)
        pop = [creator.Individual(tokenizer(random_fluent_output[i]).input_ids) for i in range(num_inputs)]
    elif start_population_as == "output":
        target_seq = prefix + assistant_text(sample_x)
        target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
        target_tokens = target_tokens[remove_prefix:]
        output = target_tokens.tolist()[:max_input_len]
        pop = [creator.Individual(output) for _ in range(num_inputs)]
    elif start_population_as == "output_paraphrase":
        output_paraphrase = paraphrase_model_generate(assistant_text(sample_x), num_paraphrase=num_inputs, max_new_tokens=max_input_len)
        pop = [creator.Individual(tokenizer(output_paraphrase[i]).input_ids) for i in range(num_inputs)]
    elif start_population_as == "output_synonym":
        output_synonym = synonyms_model_generate(assistant_text(sample_x), num_synonyms=num_inputs, max_new_tokens=max_input_len)
        pop = [creator.Individual(tokenizer(output_synonym[i]).input_ids) for i in range(num_inputs)]
    elif start_population_as == "inversion":
        inversion = tokenizer(inversion_model_generate(assistant_text(sample_x), num_inversions=1, max_new_tokens=max_input_len, sample=False)).input_ids[0]
        pop = [creator.Individual(inversion) for _ in range(num_inputs)]
    elif start_population_as == "inversion_sample":
        inversion_sample = inversion_model_generate(assistant_text(sample_x), num_inversions=num_inputs, max_new_tokens=max_input_len, sample=True)
        pop = [creator.Individual(tokenizer(inversion_sample[i]).input_ids) for i in range(num_inputs)]

    return pop

##### ~Main loop - for individual~

In [ ]:
# # Parameters which are manually defined
# target_seq = "Your face is ugly"
# max_input_len = 15
# num_inputs = 1000
# total_gens = 200
# model_name = "gpt2"
# # embedding_size

# # Parameters which are model specific
# if model_name == "gpt2":
#     in_batch_size = 400
#     prefix = " "
#     remove_prefix = 0
#     vocab_size = 50257
#     prompt_template = lambda x: f"User: {x}\nAssistant:"
# if model_name == "llama2":
#     in_batch_size = 200
#     prefix = "\n\n"
#     remove_prefix = 4
#     vocab_size = 32000
#     prompt_template = lambda x: f"[INST]{x}[/INST]\n\n"

# # Tokenize target sequence
# target_seq = prefix + target_seq
# target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
# target_tokens = target_tokens[remove_prefix:]

In [ ]:
# # Define population
# creator.create("FitnessMax", base.Fitness, weights=(1.0,))
# creator.create("Individual", list, fitness=creator.FitnessMax)
# toolbox = base.Toolbox()
# toolbox.register("attr_bool", random.randint, 0, vocab_size-1)
# toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool)

# # Define algorithm operations
# toolbox.register("evaluate", eval_inds, in_batch_size=in_batch_size, target_tokens=target_tokens)
# toolbox.register("mate", tools.cxUniform, indpb=0.3)
# toolbox.register("mutate", custom_mutate, indpb=0.1, bounded=False)
# toolbox.register("select", tools.selTournament, tournsize=15)

# # Set up recorded statistics
# halloffame = tools.HallOfFame(maxsize=10)
# stats = tools.Statistics(key=lambda ind: ind.fitness.values)
# stats.register("max", numpy.max)
# stats.register("min", numpy.min)
# stats.register("avg", numpy.mean)
# stats.register("std", numpy.std)

In [ ]:
# # Run algorithm (and print proggress stats)
# pop = [toolbox.individual(n=random.randint(1,max_input_len)) for _ in range(num_inputs)]
# pop, logbook = eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=total_gens, halloffame=halloffame, stats=stats, verbose=True)

# # Print best results
# print(tokenizer.decode(tools.selBest(pop, k=1)[0]))
# print(tokenizer.batch_decode(halloffame))

In [ ]:
# books = []
# for aaa in range(5):
#     # Run algorithm (and print proggress stats)
#     pop = [toolbox.individual(n=random.randint(1,max_input_len)) for _ in range(num_inputs)]
#     pop, logbook = eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=total_gens, halloffame=halloffame, stats=stats, verbose=True)
#     books.append(logbook)

#     # Print best results
#     print(tokenizer.decode(tools.selBest(pop, k=1)[0]))
#     print(tokenizer.batch_decode(halloffame))

In [ ]:
# # CODE FOR CHECKPOINTING
# # Set algorithm parameters
# total_gens = 200
# checkpoint_freq = 10

# # Recover population from checkpoint if want
# # cp = pickle.load(open("ga_checkpoint.pkl", "r"))
# # pop = cp["population"]
# # g = cp["generation"]
# # random.setstate(cp["rndstate"])

# # Otherwise initialise populisation
# pop = [toolbox.individual(n=random.randint(1,max_input_len)) for _ in range()]
# g = 0

# # Run algorithm (and print proggress stats)
# split_gens = (total_gens - g) // checkpoint_freq
# for i in range(split_gens):
#     pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=checkpoint_freq, halloffame=halloffame, stats=stats verbose=True)

# Save checkpoint if want
# g = 3000
# cp = dict(population=pop, generation=g, rndstate=random.getstate(), logbook=logbook)
# pickle.dump(cp, open("/content/drive/MyDrive/ga_checkpoint3.pkl", "wb"), 2)

#### Main loop - for dataset

In [ ]:
# Run/ search params
max_input_len = 15 # Doesnt matter since bounded=False
num_inputs = 1000
max_gens = 50#200 #999 # unbounded
max_minutes = None#10
stop_early = True

# Initialisation params
start_population_as = [
    "random",
    "random_dataset",
    "random_fluent",
    "random_fluent_output",
    "output",
    "output_paraphrase",
    "output_synonym",
    "inversion",
    "inversion_sample"
][0]

initial_model_name = [None, "gpt2", "gpt2-large", "llama2"][0]
initial_model_based_on = ["gens", "max_score"][1]
initial_model_param = [[10, 50, 100], [0.01, 0.1]][1][1]

# Model params
main_model_name = ["gpt2", "llama2"][0]
if initial_model_name is None:
    set_model_params(main_model_name)
    using_main_model = True
else:
    set_model_params(initial_model_name)
    using_main_model = False

# Define population
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, vocab_size-1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool)

# Define algorithm operations
toolbox.register("mate", tools.cxUniform, indpb=0.3)
toolbox.register("mutate", custom_mutate, indpb=0.1, bounded=False)
toolbox.register("select", tools.selTournament, tournsize=15)

# Set up recorded statistics
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("max", numpy.max)
stats.register("min", numpy.min)
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)

In [ ]:
def normal_ga(start_arg, end_arg):
    for x in range(start_arg, end_arg):
        experiment_start_time = time.process_time()#datetime.now()

        print("\n\n"+"-"*50+"\n"+"-"*50+"\nOriginal input:", [user_text(x)])
        print("Target output:", [assistant_text(x)])
        print("\nCurrent target output:", [assistant_text(x)])
        print("gen	nevals	time          	max        	min	avg        	std        ")

        # Tokenize target sequence
        target_seq = prefix + assistant_text(x)
        target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
        target_tokens = target_tokens[remove_prefix:]

        # Eval function and hall of fame need to be reset between runs
        toolbox.register("evaluate", eval_inds, in_batch_size=in_batch_size, target_tokens=target_tokens, prompt_template=prompt_template)
        halloffame = tools.HallOfFame(maxsize=10)

        # Initalise population
        pop = initialise_text_pop(start_population_as, x, max_input_len, num_inputs)

        # Run algorithm (and print proggress stats)
        pop, logbook, out_of_time = eaSimple(
            pop,
            toolbox,
            cxpb=0.5,
            mutpb=0.2,
            ngen=max_gens,
            halloffame=halloffame,
            stats=stats,
            verbose=True,
            experiment_start_time=experiment_start_time,
            max_minutes=max_minutes,
            stop_early=True,
        )

        if out_of_time:
            continue

In [ ]:
start_population_as = [
    "random",
    "random_dataset",
    "random_fluent",
    "random_fluent_output",
    "output",
    "output_paraphrase",
    "output_synonym",
    "inversion",
    "inversion_sample"
][6]

In [ ]:
normal_ga(0, 10)

In [ ]:
normal_ga(10, 20)

In [ ]:
normal_ga(20, 30)

#### Main loop - for dataset (proggressive)

In [ ]:
# Run/ search params
max_input_len = 15  # Doesnt matter since bounded=False
num_inputs = 100#1000
max_gens = 40#200 #999 # unbounded
max_minutes = None#10
stop_early = True

# Initialisation params
start_population_as = [
    "random",
    "random_dataset",
    "random_fluent",
    "random_fluent_output",
    "output",
    "output_paraphrase",
    "output_synonym",
    "inversion",
    "inversion_sample"
][2]

initial_model_name = [None, "gpt2", "gpt2-large", "llama2"][1]
initial_model_based_on = ["gens", "max_score"][1]
initial_model_param = [[50, 100], [0.01, 0.1]][0][0]

# Model params
main_model_name = ["gpt2", "llama2"][0]
if initial_model_name is None:
    set_model_params(main_model_name)
    using_main_model = True
else:
    set_model_params(initial_model_name)
    using_main_model = False

# Proggressive params
progressive_based_on = ["gens", "non_zero_scores"][0]
progressive_param = 1#200

# Define population
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, vocab_size-1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool)

# Define algorithm operations
toolbox.register("mate", tools.cxUniform, indpb=0.3)
toolbox.register("mutate", custom_mutate, indpb=0.1, bounded=False)
toolbox.register("select", tools.selTournament, tournsize=15)

# Set up recorded statistics
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("max", numpy.max)
stats.register("min", numpy.min)
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)

In [ ]:
def proggressive_nzs_ga(start_arg, end_arg):
    for x in range(start_arg, end_arg):
        experiment_start_time = time.process_time()#datetime.now()

        print("\n\n"+"-"*50+"\n"+"-"*50+"\nOriginal input:", [user_text(x)])
        print("Target output:", [assistant_text(x)])

        # Tokenize target sequence
        target_seq = prefix + assistant_text(x)
        target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
        target_tokens = target_tokens[remove_prefix:]

        # Things reset between examples from dataset
        halloffame = tools.HallOfFame(maxsize=10)

        # Initalise population
        pop = initialise_text_pop(start_population_as, x, max_input_len, num_inputs)

        # Initialise current target output
        target_tok_id = 0
        target_updated = False
        run_target_tokens = target_tokens[:target_tok_id+1]
        print("\nCurrent target output:", [tokenizer.decode(run_target_tokens)])
        print("gen	nevals	time          	max        	min	avg        	std        ")
        toolbox.register("evaluate", eval_inds, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template)

        # Run search until out of resources
        gens_so_far = 0
        out_of_time = False
        while not ((gens_so_far >= max_gens) or out_of_time):
            if target_updated:
                # Update current target output
                run_target_tokens = target_tokens[:target_tok_id+1]
                print("\nCurrent target output:", [tokenizer.decode(run_target_tokens)])
                print("gen	nevals	time          	max        	min	avg        	std        ")
                toolbox.register("evaluate", eval_inds, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template)

                # All fitness functions need to be updated based on the new target output (if theyre non zero)
                # Smart method of having the probabilities be updated only by one extra target token
                invalid_ind = [ind for ind in pop if (not ind.fitness.valid) or (ind.fitness.values[0] != 0)]
                fitnesses = eval_inds(ind_list=invalid_ind, target_tok_id=target_tok_id, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template)
                for ind, fit in zip(invalid_ind, fitnesses):
                    ind.fitness.values = fit

                # Update the stats based on this partial update
                record = stats.compile(pop) if stats else {}
                logbook = tools.Logbook()
                logbook.header = ['gen', 'nevals', 'time'] + (stats.fields if stats else [])
                gen_addition = 0.5
                logbook.record(gen=(gens_so_far+gen_addition), nevals=len(invalid_ind), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **record)
                print(logbook.__str__().split("\n")[-1])
                # Print best ones
                best = tools.selBest(pop, k=1000)
                best_fitnesses = [ind.fitness.values[0] for ind in best]
                print([[best_fitnesses[i], [tokenizer.decode(best[i]).replace("'", "\\'").replace('"', '\\"')]] for i in range(len(best))])

                target_updated = False

            # Run genetic algo for one step
            pop, logbook, out_of_time = eaSimple(
                pop,
                toolbox,
                cxpb=0.5,
                mutpb=0.2,
                ngen=1,
                halloffame=halloffame,
                stats=stats, verbose=True,
                gens_so_far=gens_so_far,
                experiment_start_time=experiment_start_time,
                max_minutes=max_minutes,
                stop_early=True,
            )
            gens_so_far += 1

            # Check if should start considering more tokens, if percentage of non zero values in population are high enough
            if (gens_so_far < max_gens) and (len(run_target_tokens) != len(target_tokens)):
                non_zeros = [1 for ind in pop if ind.fitness.values[0] != 0]
                non_zeros_percent = ((sum(non_zeros)/num_inputs)*100)
                if non_zeros_percent >= progressive_param:
                    target_tok_id += len(target_tokens) // 10 # Start considering next 10% of tokens
                    target_updated = True


In [ ]:
proggressive_nzs_ga(6,7)

In [ ]:
def proggressive_gens_ga(start_arg, end_arg):
    for x in range(start_arg, end_arg):
        experiment_start_time = time.process_time()#datetime.now()

        print("\n\n"+"-"*50+"\n"+"-"*50+"\nOriginal input:", [user_text(x)])
        print("Target output:", [assistant_text(x)])

        # Tokenize target sequence
        target_seq = prefix + assistant_text(x)
        target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
        target_tokens = target_tokens[remove_prefix:]

        # Things reset between examples from dataset
        halloffame = tools.HallOfFame(maxsize=10)

        # Initalise population
        pop = initialise_text_pop(start_population_as, x, max_input_len, num_inputs)

        # Decide on progressive intervals over the total number of generations
        gen_slices = [progressive_param // len(target_tokens) for _ in range(len(target_tokens))]
        # Spread the leftover generations evenly starting from final slice and going backwards until run out (earlier slices likely won't get any extra)
        for slice_i in range(len(gen_slices)-1, (len(gen_slices)-1-(progressive_param % len(target_tokens))), -1):
            gen_slices[slice_i] += 1
        assert sum(gen_slices) == progressive_param

        for gen_i in range(len(gen_slices)):
            # gen_slice is the amount of generations to currently do
            gen_slice = gen_slices[gen_i]
            if gen_slice == 0:
                continue
            gens_so_far = sum(gen_slices[:gen_i])

            # Reveal another token of the target output for each gen_slice completed
            run_target_tokens = target_tokens[:gen_i+1]
            print("\nCurrent target output:", [tokenizer.decode(run_target_tokens)])
            print("gen	nevals	time          	max        	min	avg        	std        ")

            if gens_so_far != 0:
                # All fitness functions need to be updated based on the new target output (if theyre non zero)
                # Smart method of having the probabilities be updated only by one extra target token
                invalid_ind = [ind for ind in pop if (not ind.fitness.valid) or (ind.fitness.values[0] != 0)]
                fitnesses = eval_inds(ind_list=invalid_ind, target_tok_id=gen_i, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template)
                for ind, fit in zip(invalid_ind, fitnesses):
                    ind.fitness.values = fit

                # Update the stats based on this partial update
                record = stats.compile(pop) if stats else {}
                logbook = tools.Logbook()
                logbook.header = ['gen', 'nevals', 'time'] + (stats.fields if stats else [])
                gen_addition = 0.5
                logbook.record(gen=(gens_so_far+gen_addition), nevals=len(invalid_ind), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **record)
                print(logbook.__str__().split("\n")[-1])
                # Print best ones
                best = tools.selBest(pop, k=1000)
                best_fitnesses = [ind.fitness.values[0] for ind in best]
                print([[best_fitnesses[i], [tokenizer.decode(best[i]).replace("'", "\\'").replace('"', '\\"')]] for i in range(len(best))])

            # Run genetic algo with new eval function
            toolbox.register("evaluate", eval_inds, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template)
            breaking = False
            for g in range(gen_slice):
                pop, logbook, out_of_time = eaSimple(
                    pop,
                    toolbox,
                    cxpb=0.5,
                    mutpb=0.2,
                    ngen=1,
                    halloffame=halloffame,
                    stats=stats,
                    verbose=True,
                    gens_so_far=gens_so_far,
                    experiment_start_time=experiment_start_time,
                    max_minutes=max_minutes,
                    stop_early=True,
                )
                gens_so_far += 1
                if (gens_so_far >= max_gens) or out_of_time:
                    breaking = True
                    break
            if breaking:
                break

        # If still havent reached max gens/ time then continue searching with full target output
        for g in range(gens_so_far,max_gens):
            if out_of_time:
                break

            # Run genetic algo with new eval function
            # toolbox.register("evaluate", eval_inds, in_batch_size=in_batch_size, target_tokens=target_tokens, prompt_template=prompt_template)
            pop, logbook, out_of_time = eaSimple(
                pop,
                toolbox,
                cxpb=0.5,
                mutpb=0.2,
                ngen=1,
                halloffame=halloffame,
                stats=stats, verbose=True,
                gens_so_far=g,
                experiment_start_time=experiment_start_time,
                max_minutes=max_minutes,
                stop_early=True,
            )


In [ ]:
proggressive_gens_ga(6,7)

### Embedding-based Optimisation [PSO (using DEAP)]

In [ ]:
%%capture
!pip install deap

In [ ]:
import operator
import random
import pickle

import numpy
import math

import torch.nn.functional as F
import torch
from deap import base, creator, tools

import gc

from datetime import datetime, timedelta
import time

# transformers.logging.set_verbosity_error()

# from deap import benchmarks

#### Set up

##### Algo definitions

In [ ]:
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Particle", list, fitness=creator.FitnessMax, speed=list,
    smin=None, smax=None, best=None)

In [ ]:
def generate(size, pmin, pmax, smin, smax):
    part = creator.Particle(random.uniform(pmin, pmax) for _ in range(size))
    part.speed = [random.uniform(smin, smax) for _ in range(size)]
    part.smin = smin
    part.smax = smax
    return part

def generate_static(size, smin, smax, p_static):
    part = creator.Particle(p_static)
    part.speed = [random.uniform(smin, smax) for _ in range(size)]
    part.smin = smin
    part.smax = smax
    return part

def updateParticle(part, best, phi1, phi2):
    u1 = (random.uniform(0, phi1) for _ in range(len(part)))
    u2 = (random.uniform(0, phi2) for _ in range(len(part)))
    v_u1 = map(operator.mul, u1, map(operator.sub, part.best, part))
    v_u2 = map(operator.mul, u2, map(operator.sub, best, part))
    part.speed = list(map(operator.add, part.speed, map(operator.add, v_u1, v_u2)))
    for i, speed in enumerate(part.speed):
        if abs(speed) < part.smin:
            part.speed[i] = math.copysign(part.smin, speed)
        elif abs(speed) > part.smax:
            part.speed[i] = math.copysign(part.smax, speed)
    part[:] = list(map(operator.add, part, part.speed))

##### Evaluation function

In [ ]:
def eval_inds(emb_list, in_batch_size, target_tokens, t5_batch_size, prompt_template=None, max_input_len=15, target_tok_id=0, ind_evals=None):
    # global autoencoder

    # Have to convert to list of strings to stay same with previous code
    emb_list_list = [list([e]) for e in emb_list]
    emb_tensor = torch.Tensor(emb_list_list).to("cuda")
    string_list = t5_decode(emb_tensor, max_out_length=max_input_len, t5_batch_size=t5_batch_size)
    # del emb_tensor
    # # del autoencoder
    # gc.collect()
    # torch.cuda.empty_cache()
    # gc.collect()
    # torch.cuda.empty_cache()
    # # autoencoder = BottleneckT5Autoencoder(model_path='thesephist/contra-bottleneck-t5-small-wikipedia', device=device)
    # print("cleaned")

    if prompt_template is not None:
        string_list = [prompt_template(s) for s in string_list]

    num_strings = len(string_list)
    batch_size = min(in_batch_size, num_strings)
    # Initialise batch
    batch_string_ids = list(range(0,batch_size))
    next_string_to_batch = batch_size

    # Initialise string trackers
    if target_tok_id == 0 or ind_evals == None:
        string_probabilities = [1]*num_strings
        string_target_tok_id = [0]*num_strings
    else:
        # Individuals have already gotten probabilities for some tokens
        string_probabilities = [eval for eval in ind_evals]
        string_target_tok_id = [target_tok_id for _ in ind_evals]

    successful_string_ids = []

    # Go until there are no more strings being checked nor yet to be checked
    empty_batch_slots = 0
    counter = 0
    while empty_batch_slots != batch_size:
        # Get batch ready for inference
        batch_strings = []
        for b_id in range(batch_size):
            if b_id >= num_strings:
                break
            s_id = batch_string_ids[b_id]
            if s_id == -1:
                continue
            # Need to add target tokens checked so far to end of string
            b_string = string_list[batch_string_ids[b_id]]
            b_string += tokenizer.decode(target_tokens[:string_target_tok_id[s_id]])
            batch_strings.append(b_string)

        # Do the inference
        input_ids = tokenizer(batch_strings, return_tensors="pt", padding=True).to("cuda")
        generate_output = model.generate(**input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=300, max_new_tokens=1, return_dict_in_generate=True, output_scores=True, pad_token_id=eos_token,)
        # generate_output = model.generate(**input_ids, temperature=0.7, do_sample=True, top_p=0.95, top_k=40, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)

        # Update validity of strings based on inference
        empty = 0
        for b_id in range(batch_size):
            s_id = batch_string_ids[b_id]
            if s_id == -1:
                empty += 1
                continue
            curr_target_tok = target_tokens[string_target_tok_id[s_id]]
            prob = generate_output.scores[0][b_id-empty][curr_target_tok]

            if not (math.isinf(prob) and prob < 0):
                # String still viable candidate
                # Update probability
                True_distribution = F.softmax(generate_output.scores[0][b_id-empty], dim=0)
                string_probabilities[s_id] *= float(True_distribution[curr_target_tok])

                # Prepare to check for next target token
                string_target_tok_id[s_id] += 1
                if string_target_tok_id[s_id] == len(target_tokens):
                    # Have already checked all the target tokens
                    successful_string_ids.append(s_id)
                    # Finished so pretend string is not viable so it gets replaced
                    prob = -math.inf
            else:
                # String not a viable candidate
                # Update probability
                string_probabilities[s_id] = 0

            if math.isinf(prob) and prob < 0:
                # Replace non-viable and finished strings in batch
                if next_string_to_batch >= num_strings:
                    # No new strings to start checking
                    batch_string_ids[b_id] = -1
                    empty_batch_slots += 1
                else:
                    batch_string_ids[b_id] = next_string_to_batch
                    next_string_to_batch += 1
        counter += 1

    for part, prob in zip(emb_list,string_probabilities):
        part.fitness.values = (prob,)

##### Initialisation function

In [ ]:
def initialise_embed_pop(start_population_as, sample_x, max_input_len, num_inputs):
    if start_population_as == "random":
        pop = toolbox.population(n=num_inputs)
    elif start_population_as == "random_dataset":
        pop = []
        random_dataset = rand_data_model_generate(num_data=num_inputs, max_new_tokens=max_input_len)
        for n in range(num_inputs):
            one_embedding = t5_encode(random_dataset[n]).tolist()[0]
            pop.append(toolbox.particle_static(p_static=one_embedding))
    elif start_population_as == "random_fluent":
        pop = []
        random_fluent = fluent_model_generate(num_generate=num_inputs, max_new_tokens=max_input_len)
        for n in range(num_inputs):
            one_embedding = t5_encode(random_fluent[n]).tolist()[0]
            pop.append(toolbox.particle_static(p_static=one_embedding))
    elif start_population_as == "random_fluent_output":
        pop = []
        random_fluent_output = fluent_model_generate(seed=assistant_text(sample_x), num_generate=num_inputs, max_new_tokens=max_input_len)
        for n in range(num_inputs):
            one_embedding = t5_encode(random_fluent_output[n]).tolist()[0]
            pop.append(toolbox.particle_static(p_static=one_embedding))
    elif start_population_as == "output":
        pop = []
        one_embedding = t5_encode(assistant_text(sample_x)).tolist()[0]
        for n in range(num_inputs):
            pop.append(toolbox.particle_static(p_static=one_embedding))
    elif start_population_as == "output_paraphrase":
        pop = []
        output_paraphrase = paraphrase_model_generate(assistant_text(sample_x), num_paraphrase=num_inputs, max_new_tokens=max_input_len)
        for n in range(num_inputs):
            one_embedding = t5_encode(output_paraphrase[n]).tolist()[0]
            pop.append(toolbox.particle_static(p_static=one_embedding))
    elif start_population_as == "output_synonym":
        pop = []
        output_synonym = synonyms_model_generate(assistant_text(sample_x), num_synonyms=num_inputs, max_new_tokens=max_input_len)
        for n in range(num_inputs):
            one_embedding = t5_encode(output_synonym[n]).tolist()[0]
            pop.append(toolbox.particle_static(p_static=one_embedding))
    elif start_population_as == "inversion":
        pop = []
        inversion = inversion_model_generate(assistant_text(sample_x), num_inversions=1, max_new_tokens=max_input_len, sample=False)[0]
        one_embedding = t5_encode(inversion).tolist()[0]
        for n in range(num_inputs):
            pop.append(toolbox.particle_static(p_static=one_embedding))
    elif start_population_as == "inversion_sample":
        pop = []
        inversion_sample = inversion_model_generate(assistant_text(sample_x), num_inversions=num_inputs, max_new_tokens=max_input_len, sample=True)
        for n in range(num_inputs):
            one_embedding = t5_encode(inversion_sample[n]).tolist()[0]
            pop.append(toolbox.particle_static(p_static=one_embedding))

    return pop

##### ~Main loop - for individual~

In [ ]:
# # Parameters which are manually defined
# target_seq = "Your face is ugly"
# max_input_len = 64#999 # Basically unbounded?
# num_inputs = 1000
# total_gens = 200

# # Parameters which are model specific
# t5_batch_size = 335 #500
# model_name = "gpt2"

# if model_name == "gpt2":
#     in_batch_size = 200
#     prefix = " "
#     remove_prefix = 0
#     vocab_size = 50257
#     prompt_template = lambda x: f"User: {x}\nAssistant:"
# if model_name == "llama2":
#     in_batch_size = 200
#     prefix = "\n\n"
#     remove_prefix = 4
#     vocab_size = 32000
#     prompt_template = lambda x: f"[INST]{x}[/INST]\n\n"

# # Tokenize target sequence
# target_seq = prefix + target_seq
# target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
# target_tokens = target_tokens[remove_prefix:]

# # ------

# toolbox = base.Toolbox()
# toolbox.register("particle", generate, size=embedding_size, pmin=-1, pmax=1, smin=-0.5, smax=0.5)
# toolbox.register("population", tools.initRepeat, list, toolbox.particle)
# toolbox.register("update", updateParticle, phi1=2.0, phi2=2.0)
# # toolbox.register("evaluate", benchmarks.h1)
# toolbox.register("evaluate", eval_inds, t5_batch_size=t5_batch_size, in_batch_size=in_batch_size, target_tokens=target_tokens, prompt_template=prompt_template, max_input_len=max_input_len)

# stats = tools.Statistics(lambda ind: ind.fitness.values)
# stats.register("max", numpy.max)
# stats.register("min", numpy.min)
# stats.register("avg", numpy.mean)
# stats.register("std", numpy.std)

# from_checkpoint = False
# if not from_checkpoint:
#     pop = toolbox.population(n=num_inputs)
#     g_start = 0
#     logbook = tools.Logbook()
#     logbook.header = ["gen", "evals"] + stats.fields
#     best = None
# # else:
# #     cp = pickle.load(open("/content/drive/MyDrive/pso_checkpoint1.pkl", "rb"))
# #     pop = cp["population"]
# #     g_start = cp["generation"]
# #     logbook = cp["logbook"]
# #     random.setstate(cp["rndstate"])
# #     best = cp["best"]

# for g in range(g_start, total_gens):
#     toolbox.evaluate(emb_list=pop)
#     for part in pop:
#         # part.fitness.values = toolbox.evaluate(part)
#         if not part.best or part.best.fitness < part.fitness:
#             part.best = creator.Particle(part)
#             part.best.fitness.values = part.fitness.values
#         if not best or best.fitness < part.fitness:
#             best = creator.Particle(part)
#             best.fitness.values = part.fitness.values
#     for part in pop:
#         toolbox.update(part, best)

#     # Gather all the fitnesses in one list and print the stats
#     logbook.record(gen=g, evals=len(pop), **stats.compile(pop))
#     print("")
#     print(logbook.stream)
#     print("Best so far:", [t5_decode(torch.unsqueeze(torch.Tensor(tools.selBest(pop, k=1)[0]), dim=0).to("cuda"), max_out_length=max_input_len, t5_batch_size=1)])

#     # if g % 10 == 0:
#     #     cp = dict(population=pop, generation=g, rndstate=random.getstate(), logbook=logbook, best=best)
#     #     pickle.dump(cp, open("/content/drive/MyDrive/pso_checkpoint1.pkl", "wb"), 2)


In [ ]:
# toolbox = base.Toolbox()
# toolbox.register("particle", generate, size=embedding_size, pmin=-1, pmax=1, smin=-0.5, smax=0.5)
# toolbox.register("population", tools.initRepeat, list, toolbox.particle)
# toolbox.register("update", updateParticle, phi1=2.0, phi2=2.0)
# # toolbox.register("evaluate", benchmarks.h1)
# toolbox.register("evaluate", eval_inds, t5_batch_size=t5_batch_size, in_batch_size=in_batch_size, target_tokens=target_tokens, prompt_template=prompt_template, max_input_len=max_input_len)

# stats = tools.Statistics(lambda ind: ind.fitness.values)
# stats.register("max", numpy.max)
# stats.register("min", numpy.min)
# stats.register("avg", numpy.mean)
# stats.register("std", numpy.std)

# from_checkpoint = False
# if not from_checkpoint:
#     pop = toolbox.population(n=num_inputs)
#     g_start = 0
#     logbook = tools.Logbook()
#     logbook.header = ["gen", "evals"] + stats.fields
#     best = None
# # else:
# #     cp = pickle.load(open("/content/drive/MyDrive/pso_checkpoint1.pkl", "rb"))
# #     pop = cp["population"]
# #     g_start = cp["generation"]
# #     logbook = cp["logbook"]
# #     random.setstate(cp["rndstate"])
# #     best = cp["best"]

# for g in range(g_start, total_gens):
#     toolbox.evaluate(emb_list=pop)
#     for part in pop:
#         # part.fitness.values = toolbox.evaluate(part)
#         if not part.best or part.best.fitness < part.fitness:
#             part.best = creator.Particle(part)
#             part.best.fitness.values = part.fitness.values
#         if not best or best.fitness < part.fitness:
#             best = creator.Particle(part)
#             best.fitness.values = part.fitness.values
#     for part in pop:
#         toolbox.update(part, best)

#     # Gather all the fitnesses in one list and print the stats
#     logbook.record(gen=g, evals=len(pop), **stats.compile(pop))
#     print(logbook.stream)
#     print("Best so far:", [t5_decode(torch.unsqueeze(torch.Tensor(tools.selBest(pop, k=1)[0]), dim=0).to("cuda"), max_out_length=max_input_len, t5_batch_size=1)])

#     # if g % 10 == 0:
#     #     cp = dict(population=pop, generation=g, rndstate=random.getstate(), logbook=logbook, best=best)
#     #     pickle.dump(cp, open("/content/drive/MyDrive/pso_checkpoint1.pkl", "wb"), 2)


#### Main loop - for dataset

In [ ]:
# Run/ search params
max_input_len = 64 #99 # Basically unbounded?
num_inputs = 500#1000
max_gens = 50#100#200 #999 # unbounded
# max_minutes = 10
stop_early = True

# Initialisation params
start_population_as = [
    "random",
    "random_dataset",
    "random_fluent",
    "random_fluent_output",
    "output",
    "output_paraphrase",
    "output_synonym",
    "inversion",
    "inversion_sample"
][4]

initial_model_name = [None, "gpt2", "gpt2-large", "llama2"][0]
initial_model_based_on = ["gens", "max_score"][1]
initial_model_param = [[50, 100], [0.01, 0.1]][0][0]

# Model params
t5_batch_size = 100
main_model_name = ["gpt2", "llama2"][0]
if initial_model_name is None:
    set_model_params(main_model_name)
    using_main_model = True
else:
    set_model_params(initial_model_name)
    using_main_model = False

# Define algorithm operations
toolbox = base.Toolbox()
toolbox.register("particle_static", generate_static, size=embedding_size, smin=-0.5, smax=0.5) # For initialising as target output
toolbox.register("particle", generate, size=embedding_size, pmin=-0.5, pmax=0.5, smin=-0.5, smax=0.5)
toolbox.register("population", tools.initRepeat, list, toolbox.particle)
toolbox.register("update", updateParticle, phi1=2.0, phi2=2.0)

# Set up recorded statistics
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("max", numpy.max)
stats.register("min", numpy.min)
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)

In [ ]:
def normal_pso(start_arg, end_arg):
    global using_main_model

    for x in range(start_arg, end_arg):
        experiment_start_time = time.process_time()#datetime.now()

        print("\n\n"+"-"*50+"\n"+"-"*50+"\nOriginal input:", [user_text(x)])
        print("Target output:", [assistant_text(x)])
        print("\nCurrent target output:", [assistant_text(x)])
        print("gen	nevals	time          	max        	min	avg        	std        ")


        # Tokenize target sequence
        target_seq = prefix + assistant_text(x)
        target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
        target_tokens = target_tokens[remove_prefix:]

        # Update evaluate based on new target sequence
        # toolbox.register("evaluate", benchmarks.h1)
        toolbox.register("evaluate", eval_inds, t5_batch_size=t5_batch_size, in_batch_size=in_batch_size, target_tokens=target_tokens, prompt_template=prompt_template, max_input_len=max_input_len)

        # Initalise population
        pop = initialise_embed_pop(start_population_as, x, max_input_len, num_inputs)

        logbook = tools.Logbook()
        logbook.header = ["gen", "nevals", "time"] + stats.fields
        best = None

        # best_so_far = 0
        # best_so_far_text = None
        for g in range(max_gens):
            toolbox.evaluate(emb_list=pop)
            for part in pop:
                # part.fitness.values = toolbox.evaluate(part)
                if not part.best or part.best.fitness < part.fitness:
                    part.best = creator.Particle(part)
                    part.best.fitness.values = part.fitness.values
                if not best or best.fitness < part.fitness:
                    best = creator.Particle(part)
                    best.fitness.values = part.fitness.values
            for part in pop:
                toolbox.update(part, best)

            # Gather all the fitnesses in one list and print the stats
            logbook.record(gen=g+1, nevals=len(pop), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **stats.compile(pop))
            print(logbook.__str__().split("\n")[-1])
            # Print best ones
            bestest = tools.selBest(pop, k=2)#1000)
            best_fitnesses = [ind.fitness.values[0] for ind in bestest]
            best_list = [list([e]) for e in bestest]
            best_tensor = torch.Tensor(best_list).to("cuda")
            best_inds = t5_decode(torch.Tensor(best_tensor).to("cuda"), max_out_length=max_input_len, t5_batch_size=t5_batch_size)
            print([[best_fitnesses[i], [best_inds[i]]] for i in range(len(bestest))])

            # Check if should be stopping
            if stop_early and best_fitnesses[0] == 1:
                break
            # if timedelta(seconds=(time.process_time()-experiment_start_time)) >= timedelta(minutes=max_minutes):
            #     break

            # Check if should be switching from intial model to main model, if havent already
            if not using_main_model:
                if initial_model_based_on == "gens":
                    if (g+1) >= initial_model_param:
                        set_model_params(main_model_name)
                        using_main_model = True
                elif initial_model_based_on == "max_score":
                    if best_fitnesses[0] >= initial_model_param:
                        set_model_params(main_model_name)
                        using_main_model = True


In [ ]:
start_population_as = [
    "random",
    "random_dataset",
    "random_fluent",
    "random_fluent_output",
    "output",
    "output_paraphrase",
    "output_synonym",
    "inversion",
    "inversion_sample"
][1]

In [ ]:
normal_pso(0, 10)

In [ ]:
normal_pso(10, 20)

In [ ]:
normal_pso(20, 30)

#### Main loop - for dataset (proggressive)

In [ ]:
# Run/ search params
max_input_len = 15#64 #99 # Basically unbounded?
num_inputs = 50#20#1000
max_gens = 5 #200 #999 # unbounded
# max_minutes = 10
stop_early = True

# Initialisation params
start_population_as = [
    "random",
    "random_dataset",
    "random_fluent",
    "random_fluent_output",
    "output",
    "output_paraphrase",
    "output_synonym",
    "inversion",
    "inversion_sample"
][2]

initial_model_name = [None, "gpt2", "gpt2-large", "llama2"][1]
initial_model_based_on = ["gens", "max_score"][1]
initial_model_param = [[50, 100], [0.01, 0.1]][0][0]

# Model params
t5_batch_size = 200
main_model_name = ["gpt2", "llama2"][0]
if initial_model_name is None:
    set_model_params(main_model_name)
    using_main_model = True
else:
    set_model_params(initial_model_name)
    using_main_model = False

# Proggressive params
progressive_based_on = ["gens", "non_zero_scores"][1]
progressive_param = 1

# Define algorithm operations
toolbox = base.Toolbox()
toolbox.register("particle_static", generate_static, size=embedding_size, smin=-0.5, smax=0.5) # For initialising as target output
toolbox.register("particle", generate, size=embedding_size, pmin=-1, pmax=1, smin=-0.5, smax=0.5)
toolbox.register("population", tools.initRepeat, list, toolbox.particle)
toolbox.register("update", updateParticle, phi1=2.0, phi2=2.0)

# Set up recorded statistics
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("max", numpy.max)
stats.register("min", numpy.min)
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)

In [ ]:
def proggressive_nzs_pso(start_arg, end_arg):
    global using_main_model

    for x in range(start_arg, end_arg):
        experiment_start_time = time.process_time()#datetime.now()

        print("\n\n"+"-"*50+"\n"+"-"*50+"\nOriginal input:", [user_text(x)])
        print("Target output:", [assistant_text(x)])

        # Tokenize target sequence
        target_seq = prefix + assistant_text(x)
        target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
        target_tokens = target_tokens[remove_prefix:]

        # Initalise population
        pop = initialise_embed_pop(start_population_as, x, max_input_len, num_inputs)

        logbook = tools.Logbook()
        logbook.header = ["gen", "nevals", "time"] + stats.fields
        best = None

        # Initialise current target output
        target_tok_id = 0
        target_updated = False
        run_target_tokens = target_tokens[:target_tok_id+1]
        print("\nCurrent target output:", [tokenizer.decode(run_target_tokens)])
        print("gen	nevals	time          	max        	min	avg        	std        ")
        toolbox.register("evaluate", eval_inds, t5_batch_size=t5_batch_size, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template, max_input_len=max_input_len)

        # Run search until out of resources
        gens_so_far = 0
        out_of_time = False
        while not ((gens_so_far >= max_gens) or out_of_time):
            if target_updated:
                # Update current target output
                run_target_tokens = target_tokens[:target_tok_id+1]
                print("\nCurrent target output:", [tokenizer.decode(run_target_tokens)])
                print("gen	nevals	time          	max        	min	avg        	std        ")
                toolbox.register("evaluate", eval_inds, t5_batch_size=t5_batch_size, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template)

                # All fitness functions need to be updated based on the new target output (if theyre non zero)
                # Smart method of having the probabilities be updated only by one extra target token
                invalid_ind = [ind for ind in pop if (not ind.fitness.valid) or (ind.fitness.values[0] != 0)]
                eval_inds(emb_list=invalid_ind, t5_batch_size=t5_batch_size, target_tok_id=target_tok_id, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template, max_input_len=max_input_len)

                # Update the stats based on this partial update
                record = stats.compile(pop) if stats else {}
                logbook = tools.Logbook()
                logbook.header = ['gen', 'nevals', 'time'] + (stats.fields if stats else [])
                gen_addition = 0.5
                logbook.record(gen=(gens_so_far+gen_addition), nevals=len(invalid_ind), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **record)
                print(logbook.__str__().split("\n")[-1])
                # Print best ones
                bestest = tools.selBest(pop, k=1000)
                best_fitnesses = [ind.fitness.values[0] for ind in bestest]
                best_list = [list([e]) for e in bestest]
                best_tensor = torch.Tensor(best_list).to("cuda")
                best_inds = t5_decode(torch.Tensor(best_tensor).to("cuda"), max_out_length=max_input_len, t5_batch_size=t5_batch_size)
                print([[best_fitnesses[i], [best_inds[i]]] for i in range(len(bestest))])

                target_updated = False

            # Run PSO for one step
            toolbox.evaluate(emb_list=pop)
            for part in pop:
                # part.fitness.values = toolbox.evaluate(part)
                if not part.best or part.best.fitness < part.fitness:
                    part.best = creator.Particle(part)
                    part.best.fitness.values = part.fitness.values
                if not best or best.fitness < part.fitness:
                    best = creator.Particle(part)
                    best.fitness.values = part.fitness.values
            for part in pop:
                toolbox.update(part, best)

            # Gather all the fitnesses in one list and print the stats
            gens_so_far += 1
            logbook.record(gen=gens_so_far, nevals=len(pop), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **stats.compile(pop))
            # Print best ones
            print(logbook.__str__().split("\n")[-1])
            bestest = tools.selBest(pop, k=1000)
            best_fitnesses = [ind.fitness.values[0] for ind in bestest]
            best_list = [list([e]) for e in bestest]
            best_tensor = torch.Tensor(best_list).to("cuda")
            best_inds = t5_decode(torch.Tensor(best_tensor).to("cuda"), max_out_length=max_input_len, t5_batch_size=t5_batch_size)
            print([[best_fitnesses[i], [best_inds[i]]] for i in range(len(bestest))])

            # Check if should be stopping
            out_of_time = False #timedelta(seconds=(time.process_time()-experiment_start_time)) >= timedelta(minutes=max_minutes)
            out_of_time = out_of_time or (stop_early and best_fitnesses[0] == 1)

            # Check if should be switching from intial model to main model, if havent already
            if not using_main_model:
                if initial_model_based_on == "gens":
                    if gens_so_far >= initial_model_param:
                        set_model_params(main_model_name)
                        using_main_model = True
                elif initial_model_based_on == "max_score":
                    if best_fitnesses[0] >= initial_model_param:
                        set_model_params(main_model_name)
                        using_main_model = True

            # Check if should start considering more tokens, if percentage of non zero values in population are high enough
            if (gens_so_far < max_gens) and not out_of_time and (len(run_target_tokens) != len(target_tokens)):
                non_zeros = [1 for ind in pop if ind.fitness.values[0] != 0]
                non_zeros_percent = ((sum(non_zeros)/num_inputs)*100)
                if non_zeros_percent >= progressive_param:
                    target_tok_id += len(target_tokens) // 10 # Start considering next 10% of tokens
                    target_updated = True


In [ ]:
proggressive_nzs_pso(6, 7)

In [ ]:
def proggressive_gens_pso(start_arg, end_arg):
    global using_main_model

    for x in range(start_arg, end_arg):
        experiment_start_time = time.process_time()#datetime.now()

        print("\n\n"+"-"*50+"\n"+"-"*50+"\nOriginal input:", [user_text(x)])
        print("Target output:", [assistant_text(x)])

        # Tokenize target sequence
        target_seq = prefix + assistant_text(x)
        target_tokens = tokenizer(target_seq, return_tensors='pt').input_ids[0]
        target_tokens = target_tokens[remove_prefix:]

        # Initalise population
        pop = initialise_embed_pop(start_population_as, x, max_input_len, num_inputs)

        logbook = tools.Logbook()
        logbook.header = ["gen", "nevals", "time"] + stats.fields
        best = None

        # Decide on progressive intervals over the total number of generations
        gen_slices = [progressive_param // len(target_tokens) for _ in range(len(target_tokens))]
        # Spread the leftover generations evenly starting from final slice and going backwards until run out (earlier slices likely won't get any extra)
        for slice_i in range(len(gen_slices)-1, (len(gen_slices)-1-(progressive_param % len(target_tokens))), -1):
            gen_slices[slice_i] += 1
        assert sum(gen_slices) == progressive_param

        gens_so_far = 0
        # best_so_far = 0
        # best_so_far_text = None
        for gen_i in range(len(gen_slices)):
            # gen_slice is the amount of generations to currently do
            gen_slice = gen_slices[gen_i]
            if gen_slice == 0:
                continue
            gens_so_far = sum(gen_slices[:gen_i])

            # Reveal another token of the target output for each gen_slice completed
            run_target_tokens = target_tokens[:gen_i+1]
            print("\nCurrent target output:", [tokenizer.decode(run_target_tokens)])
            print("gen	nevals	time          	max        	min	avg        	std        ")

            if gens_so_far != 0:
                # All fitness functions need to be updated based on the new target output (if theyre non zero)
                # Smart method of having the probabilities be updated only by one extra target token
                invalid_ind = [ind for ind in pop if (not ind.fitness.valid) or (ind.fitness.values[0] != 0)]
                eval_inds(emb_list=invalid_ind, t5_batch_size=t5_batch_size, target_tok_id=gen_i, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template, max_input_len=max_input_len)

                # Update the stats based on this partial update
                record = stats.compile(pop) if stats else {}
                logbook = tools.Logbook()
                logbook.header = ['gen', 'nevals', 'time'] + (stats.fields if stats else [])
                gen_addition = 0.5
                logbook.record(gen=(gens_so_far+gen_addition), nevals=len(invalid_ind), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **record)
                print(logbook.__str__().split("\n")[-1])
                # Print best ones
                bestest = tools.selBest(pop, k=1000)
                best_fitnesses = [ind.fitness.values[0] for ind in bestest]
                best_list = [list([e]) for e in bestest]
                best_tensor = torch.Tensor(best_list).to("cuda")
                best_inds = t5_decode(torch.Tensor(best_tensor).to("cuda"), max_out_length=max_input_len, t5_batch_size=t5_batch_size)
                print([[best_fitnesses[i], [best_inds[i]]] for i in range(len(bestest))])

            # Update evaluate based on new target sequence
            toolbox.register("evaluate", eval_inds, t5_batch_size=t5_batch_size, in_batch_size=in_batch_size, target_tokens=run_target_tokens, prompt_template=prompt_template, max_input_len=max_input_len)
            breaking = False
            for g in range(gen_slice):
                # Run PSO for one step
                toolbox.evaluate(emb_list=pop)
                for part in pop:
                    # part.fitness.values = toolbox.evaluate(part)
                    if not part.best or part.best.fitness < part.fitness:
                        part.best = creator.Particle(part)
                        part.best.fitness.values = part.fitness.values
                    if not best or best.fitness < part.fitness:
                        best = creator.Particle(part)
                        best.fitness.values = part.fitness.values
                for part in pop:
                    toolbox.update(part, best)

                # Gather all the fitnesses in one list and print the stats
                gens_so_far += 1
                logbook.record(gen=gens_so_far, nevals=len(pop), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **stats.compile(pop))

                # Print best ones
                print(logbook.__str__().split("\n")[-1])
                bestest = tools.selBest(pop, k=1000)
                best_fitnesses = [ind.fitness.values[0] for ind in bestest]
                best_list = [list([e]) for e in bestest]
                best_tensor = torch.Tensor(best_list).to("cuda")
                best_inds = t5_decode(torch.Tensor(best_tensor).to("cuda"), max_out_length=max_input_len, t5_batch_size=t5_batch_size)
                print([[best_fitnesses[i], [best_inds[i]]] for i in range(len(bestest))])

                # Check if should be stopping
                out_of_time = False #timedelta(seconds=(time.process_time()-experiment_start_time)) >= timedelta(minutes=max_minutes)
                out_of_time = out_of_time or (stop_early and best_fitnesses[0] == 1)
                if (gens_so_far >= max_gens) or out_of_time:
                    breaking = True
                    break

                # Check if should be switching from intial model to main model, if havent already
                if not using_main_model:
                    if initial_model_based_on == "gens":
                        if gens_so_far >= initial_model_param:
                            set_model_params(main_model_name)
                            using_main_model = True
                    elif initial_model_based_on == "max_score":
                        if best_fitnesses[0] >= initial_model_param:
                            set_model_params(main_model_name)
                            using_main_model = True

            if breaking:
                break

        # If still havent reached max gens/ time then continue searching with full target output
        for g in range(gens_so_far+1,max_gens+1):
            if out_of_time:
                break

            toolbox.evaluate(emb_list=pop)
            for part in pop:
                # part.fitness.values = toolbox.evaluate(part)
                if not part.best or part.best.fitness < part.fitness:
                    part.best = creator.Particle(part)
                    part.best.fitness.values = part.fitness.values
                if not best or best.fitness < part.fitness:
                    best = creator.Particle(part)
                    best.fitness.values = part.fitness.values
            for part in pop:
                toolbox.update(part, best)

            # Gather all the fitnesses in one list and print the stats
            # gens_so_far += 1
            logbook.record(gen=g, nevals=len(pop), time=timedelta(seconds=(time.process_time()-experiment_start_time)), **stats.compile(pop))

            # Print best ones
            print(logbook.__str__().split("\n")[-1])
            bestest = tools.selBest(pop, k=1000)
            best_fitnesses = [ind.fitness.values[0] for ind in bestest]
            best_list = [list([e]) for e in bestest]
            best_tensor = torch.Tensor(best_list).to("cuda")
            best_inds = t5_decode(torch.Tensor(best_tensor).to("cuda"), max_out_length=max_input_len, t5_batch_size=t5_batch_size)
            print([[best_fitnesses[i], [best_inds[i]]] for i in range(len(bestest))])

            # Check if should be stopping
            out_of_time = False #timedelta(seconds=(time.process_time()-experiment_start_time)) >= timedelta(minutes=max_minutes)
            out_of_time = out_of_time or (stop_early and best_fitnesses[0] == 1)

            # Check if should be switching from intial model to main model, if havent already
            if not using_main_model:
                if initial_model_based_on == "gens":
                    if gens_so_far >= initial_model_param:
                        set_model_params(main_model_name)
                        using_main_model = True
                elif initial_model_based_on == "max_score":
                    if best_fitnesses[0] >= initial_model_param:
                        set_model_params(main_model_name)
                        using_main_model = True


In [ ]:
proggressive_gens_pso(6,7)